# 05 Anchor 是什么：参考模板怎样变成预测框

上一课我们知道 YOLOv5 在三个尺度上预测，每个尺度的每个位置对应三个 anchor。现在只解决一个问题：**anchor 与真实框、最终预测框有什么不同？**

Anchor 常译为“锚框”。在当前 YOLOv5 中，可以先把它理解成一个参考宽高模板。

## 1. 从“框有多宽、多高”开始

面对大小、长宽比不同的人脸，网络可以利用一组参考宽高作为预测的依据，而不是让所有预测都没有任何尺寸参考。

例如，某个尺度准备以下三个模板：

| 模板 | 参考宽度 | 参考高度 |
|---|---:|---:|
| A | 20 | 30 |
| B | 40 | 20 |
| C | 30 | 50 |

数字只是教学示例，不是当前权重的真实 anchor 数值。对应尺度的所有空间位置会使用同一套参考宽高。

网格提供空间位置参照，anchor 提供宽高参照，网络结合图像特征预测实际框。

## 2. 看模板与预测框的区别

![Anchor 参考模板与预测框](assets/05_anchor_reference.png)

左图在同一位置画了三种参考宽高，表示三种模板。它们不代表那里已经存在三个真实目标。

右图用一个模板示意最终预测：

- 参考宽高为 20 × 30。
- 模型根据图像特征，可以预测出 24 × 36 的实际框。
- 实际框的中心也可以相对于位置参照发生偏移。

因此，anchor 不会把最终框的大小和中心锁死。

“调整模板”是帮助理解的说法，不表示所有检测器都用简单加法计算宽高。当前 YOLOv5 对中心和宽高各有解码公式，这一课先不展开。

## 3. 三种框，作用完全不同

| 名称 | 从哪里来 | 用来做什么 |
|---|---|---|
| Anchor 参考模板 | 模型配置，或对训练数据框尺寸进行分析得到 | 给预测提供宽高参考 |
| 真实框 Ground Truth | 数据集中的人工标注 | 告诉模型需要学到的答案 |
| 预测框 | 网络根据图片计算，再经解码得到 | 模型当前给出的检测结果 |

同一套 anchor 可以用于许多图片；每张图的真实框随标注对象变化；预测框也会随图片和模型参数变化。

**人工标注告诉模型“答案是什么”，anchor 提供尺寸参考，模型负责预测实际结果。**

## 4. Anchor 不负责指定类别

三个 anchor 不对应三个口罩类别。宽高模板里没有“戴口罩”“没戴口罩”这些语义。

每个 anchor 对应的预测，都会同时产生框相关信息、目标存在性信息，以及三种口罩类别的信息。

当前模型每个候选预测有八个数：

$$
4+1+3=8
$$

其中 4 个与框有关，1 个与目标存在性有关，3 个与类别有关。训练阶段这些是尚待转换的网络输出，推理阶段还需相应解码和激活。

例如，参考模板 A 对应的预测可以检测戴口罩的人脸，也可以检测未戴口罩的人脸；类别由网络根据图像特征判断。

## 5. 与 YOLOv1 的两个框有什么不同

YOLOv1 原始结构每个位置准备多个直接预测框，没有采用这里这种 anchor 宽高模板。

当前 YOLOv5 每个尺度、每个位置，分别基于三个 anchor 产生候选预测。不同尺度通常配置不同尺寸范围的模板，帮助覆盖不同大小和长宽比的目标。

训练时不是简单在三个当前预测框中永远只选 IoU 最大的一个。当前实现会根据真实框与 anchor 的宽高匹配等规则分配监督，具体放到下一课讲。

Anchor 数值可以由数据统计等方法确定或检查。本课没有修改或重新计算当前模型的 anchor。

## 6. 自检

1. Anchor 是不是人工给某张图片画出的正确框？
2. Anchor 宽高为 20 × 30，最终预测框是否必须也是 20 × 30？
3. 三个 anchor 是否分别负责三种口罩类别？
4. 同一个 anchor 在不同图片中，对应的预测框是否一定相同？

<details>
<summary>参考答案</summary>

1. 不是。人工标注是 Ground Truth；anchor 是参考模板。
2. 不必。网络会基于参考和特征预测实际尺寸。
3. 不是。每个 anchor 对应的预测都有自己的类别输出。
4. 不一定。参考模板相同，但输入特征不同，输出也可能不同。

</details>

下一课：YOLOv5 怎样把一个真实框分配给相应的 anchor 和网格位置。

## 本地代码依据

- `vendor/yolov5/models/yolo.py`：Detect 保存 anchor 宽高，定义每个 anchor 的输出数量，并利用 anchor 相关信息解码框宽高。
- `vendor/yolov5/models/yolov5s.yaml`：标准三尺度 anchor 配置示例。
- 上一课已静态核对当前口罩权重的元数据：三个检测尺度，每处三个 anchor，每个预测八个输出。

本课只新增学习笔记和示意图，没有更改 anchor、训练数据、模型权重或部署服务。